In [1]:
%pip install -q pandas scikit-learn joblib fastapi uvicorn pydantic requests

In [4]:
import zipfile
import os

zip_path = "/content/titanic.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:")
    print(z.namelist()[:20])
    z.extractall("/content/titanic_data")

print("\nExtraction completed!")
print(os.listdir("/content/titanic_data"))

Files inside ZIP:
['gender_submission.csv', 'test.csv', 'train.csv']

Extraction completed!
['test.csv', 'gender_submission.csv', 'train.csv']


In [5]:
import os

for root, dirs, files in os.walk("/content/titanic_data"):
    if "train.csv" in files:
        print("Dataset location:", os.path.join(root, "train.csv"))

Dataset location: /content/titanic_data/train.csv


In [6]:

import pandas as pd
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Dataset location
DATA_PATH = "/content/titanic_data/train.csv"
OUTPUT_DIR = Path("/content/titanic-ml-deployment")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the specified Kaggle dataset
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print(df.head())

# Reserve unseen records before training or evaluation
development, unseen = train_test_split(
    df, test_size=0.10, random_state=42, stratify=df["Survived"]
)

# Separate features and target
X = development.drop(columns=["Survived"])
y = development["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

# Train once
model.fit(X_train, y_train)

# Evaluate on the held-out test set
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Save model and unseen records
joblib.dump(model, OUTPUT_DIR / "model.pkl")
unseen.drop(columns=["Survived"]).to_csv(
    OUTPUT_DIR / "unseen_records.csv", index=False
)

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump({"accuracy": float(accuracy)}, f, indent=2)

print("\nModel saved successfully!")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Unseen records:", len(unseen))
print("\nClassification report:")
print(classification_report(y_test, predictions))


Dataset shape: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8

In [7]:
import os

print(os.listdir("/content/titanic-ml-deployment"))

['model.pkl', 'unseen_records.csv', 'metrics.json']


In [8]:
import pandas as pd
import joblib

path = "/content/titanic-ml-deployment"

# Load the same trained model
model = joblib.load(f"{path}/model.pkl")

# Select one unseen passenger
records = pd.read_csv(f"{path}/unseen_records.csv")
passenger = records.iloc[[0]]

# Predict survival
prediction = int(model.predict(passenger)[0])
result = "Survived" if prediction == 1 else "Did not survive"

print("ONLINE INFERENCE")
print("-----------------")
print("Passenger details:")
print(passenger.to_string(index=False))
print("\nPredicted class:", prediction)
print("Prediction:", result)

ONLINE INFERENCE
-----------------
Passenger details:
 PassengerId  Pclass                     Name    Sex  Age  SibSp  Parch Ticket  Fare Cabin Embarked
         531       2 Quick, Miss. Phyllis May female  2.0      1      1  26360  26.0   NaN        S

Predicted class: 1
Prediction: Survived


In [9]:
import pandas as pd
import joblib

path = "/content/titanic-ml-deployment"
model = joblib.load(f"{path}/model.pkl")

records = pd.read_csv(f"{path}/unseen_records.csv").head(20)

if len(records) < 20:
    raise ValueError("There are fewer than 20 unseen passengers.")

results = records.copy()
results["Predicted_Survived"] = model.predict(records).astype(int)
results["Prediction_Label"] = results["Predicted_Survived"].map({
    1: "Survived",
    0: "Did not survive"
})

results.to_csv(f"{path}/batch_predictions.csv", index=False)

print("BATCH INFERENCE — 20 passengers")
print(results[["PassengerId", "Predicted_Survived", "Prediction_Label"]].to_string(index=False))
print("\nSaved: batch_predictions.csv")

BATCH INFERENCE — 20 passengers
 PassengerId  Predicted_Survived Prediction_Label
         531                   1         Survived
         402                   0  Did not survive
         434                   0  Did not survive
         242                   1         Survived
         774                   0  Did not survive
         715                   0  Did not survive
         245                   0  Did not survive
          41                   0  Did not survive
         474                   1         Survived
         272                   0  Did not survive
         727                   1         Survived
         571                   0  Did not survive
         443                   1         Survived
          83                   1         Survived
          85                   1         Survived
         158                   0  Did not survive
         398                   0  Did not survive
         502                   1         Survived
         702      

In [10]:
import pandas as pd
import joblib
import time

path = "/content/titanic-ml-deployment"
model = joblib.load(f"{path}/model.pkl")
records = pd.read_csv(f"{path}/unseen_records.csv").head(10)

print("STREAMING INFERENCE")
print("-------------------")

for i, (_, passenger) in enumerate(records.iterrows(), start=1):
    prediction = int(model.predict(passenger.to_frame().T)[0])
    result = "Survived" if prediction == 1 else "Did not survive"

    print(
        f"Record {i}/10 | Passenger ID: "
        f"{passenger['PassengerId']} | Prediction: {result}"
    )

    if i < len(records):
        time.sleep(1)

STREAMING INFERENCE
-------------------
Record 1/10 | Passenger ID: 531 | Prediction: Survived
Record 2/10 | Passenger ID: 402 | Prediction: Did not survive
Record 3/10 | Passenger ID: 434 | Prediction: Did not survive
Record 4/10 | Passenger ID: 242 | Prediction: Survived
Record 5/10 | Passenger ID: 774 | Prediction: Did not survive
Record 6/10 | Passenger ID: 715 | Prediction: Did not survive
Record 7/10 | Passenger ID: 245 | Prediction: Did not survive
Record 8/10 | Passenger ID: 41 | Prediction: Did not survive
Record 9/10 | Passenger ID: 474 | Prediction: Survived
Record 10/10 | Passenger ID: 272 | Prediction: Did not survive


In [12]:
!pip install -q fastapi uvicorn pyngrok requests

In [15]:
from pyngrok import ngrok

ngrok.kill()
print("Old tunnel stopped.")

Old tunnel stopped.


In [16]:
import subprocess
import time
import requests
import os

project = "/content/titanic-ml-deployment"

# Start FastAPI in the background
log_file = open("/content/fastapi.log", "w")

process = subprocess.Popen(
    [
        "uvicorn", "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd=project,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for the API to start
for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        print("Status:", response.status_code)
        print("Response:", response.text)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    print("FastAPI failed to start. Check its logs:")
    print(open("/content/fastapi.log").read())

FastAPI failed to start. Check its logs:
ERROR:    Error loading ASGI app. Could not import module "app".



In [17]:
import os

project = "/content/titanic-ml-deployment"

print("Folder exists:", os.path.exists(project))
print("Files in project folder:")

if os.path.exists(project):
    print(os.listdir(project))

Folder exists: True
Files in project folder:
['model.pkl', 'unseen_records.csv', 'metrics.json', 'batch_predictions.csv']


In [18]:

%%writefile /content/titanic-ml-deployment/app.py
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field, ConfigDict

BASE = Path(__file__).resolve().parent
model = joblib.load(BASE / "model.pkl")

app = FastAPI(title="Titanic Survival Prediction API")

class Passenger(BaseModel):
    model_config = ConfigDict(extra="forbid")

    Pclass: int = Field(ge=1, le=3)
    Sex: Literal["male", "female"]
    Age: float = Field(ge=0, le=100)
    SibSp: int = Field(ge=0, le=10)
    Parch: int = Field(ge=0, le=10)
    Fare: float = Field(ge=0, le=600)
    Embarked: Literal["C", "Q", "S"]

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True}

@app.post("/predict")
def predict(passenger: Passenger):
    features = pd.DataFrame([passenger.model_dump()])
    prediction = int(model.predict(features)[0])

    return {
        "prediction": prediction,
        "label": "Survived" if prediction == 1 else "Did not survive"
    }


Writing /content/titanic-ml-deployment/app.py


In [19]:
import os

project = "/content/titanic-ml-deployment"

for name in ["app.py", "model.pkl"]:
    path = os.path.join(project, name)
    print(name, "exists:", os.path.isfile(path))

app.py exists: True
model.pkl exists: True


In [20]:
import subprocess
import time
import requests

project = "/content/titanic-ml-deployment"

process = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=project,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(15):
    if process.poll() is not None:
        print("FastAPI startup error:")
        print(process.stdout.read())
        break

    try:
        response = requests.get(
            "http://127.0.0.1:8000/health", timeout=2
        )
        print("Status:", response.status_code)
        print("Response:", response.text)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    print("Server is still starting. Try the health check again.")

Status: 200
Response: {"status":"ok","model_loaded":true}


In [21]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(
    addr="http://127.0.0.1:8000",
    proto="http"
).public_url

print("API URL:", public_url)
print("Swagger UI:", public_url + "/docs")

API URL: https://a793-35-231-211-213.ngrok-free.app
Swagger UI: https://a793-35-231-211-213.ngrok-free.app/docs


In [22]:
%cd /content/titanic-ml-deployment
!python client.py

/content/titanic-ml-deployment
python3: can't open file '/content/titanic-ml-deployment/client.py': [Errno 2] No such file or directory


In [23]:

%%writefile /content/titanic-ml-deployment/client.py
import requests

BASE_URL = input("Enter your FastAPI base URL: ").strip().rstrip("/")

passenger = {
    "Pclass": 3,
    "Sex": "female",
    "Age": 22,
    "SibSp": 1,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

try:
    health = requests.get(f"{BASE_URL}/health", timeout=15)
    print("Health status:", health.status_code)
    print("Health response:", health.json())

    response = requests.post(
        f"{BASE_URL}/predict",
        json=passenger,
        timeout=15
    )
    print("Prediction status:", response.status_code)
    print("Prediction response:", response.json())
    response.raise_for_status()

except requests.RequestException as error:
    print("API request failed:", error)


Writing /content/titanic-ml-deployment/client.py


In [25]:
%cd /content/titanic-ml-deployment
!python client.py

/content/titanic-ml-deployment
Enter your FastAPI base URL: https://a793-35-231-211-213.ngrok-free.app
Health status: 200
Health response: {'status': 'ok', 'model_loaded': True}
Prediction status: 200
Prediction response: {'prediction': 1, 'label': 'Survived'}


In [26]:
%%writefile /content/titanic-ml-deployment/requirements.txt
pandas
numpy
scikit-learn
joblib
fastapi
uvicorn
pydantic
requests
pyngrok

Writing /content/titanic-ml-deployment/requirements.txt


In [27]:
from google.colab import files

files.download("/content/titanic-ml-deployment/requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>